In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import os
os.getcwd()

'D:\\ml-portfolio\\saas-customer-churn-prediction\\notebooks'

In [3]:
df=pd.read_excel("../data/Telco_customer_churn.xlsx")

In [4]:
df.columns

Index(['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code',
       'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen',
       'Partner', 'Dependents', 'Tenure Months', 'Phone Service',
       'Multiple Lines', 'Internet Service', 'Online Security',
       'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
       'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
       'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value',
       'Churn Score', 'CLTV', 'Churn Reason'],
      dtype='object')

In [5]:
df["Churn Value"].value_counts()

Churn Value
0    5174
1    1869
Name: count, dtype: int64

In [6]:
df["Churn Value"].value_counts(normalize=True)

Churn Value
0    0.73463
1    0.26537
Name: proportion, dtype: float64

In [7]:
df.isnull().sum()

CustomerID              0
Count                   0
Country                 0
State                   0
City                    0
Zip Code                0
Lat Long                0
Latitude                0
Longitude               0
Gender                  0
Senior Citizen          0
Partner                 0
Dependents              0
Tenure Months           0
Phone Service           0
Multiple Lines          0
Internet Service        0
Online Security         0
Online Backup           0
Device Protection       0
Tech Support            0
Streaming TV            0
Streaming Movies        0
Contract                0
Paperless Billing       0
Payment Method          0
Monthly Charges         0
Total Charges           0
Churn Label             0
Churn Value             0
Churn Score             0
CLTV                    0
Churn Reason         5174
dtype: int64

In [8]:
y=df["Churn Value"]

In [9]:
X=df.drop(columns=["Churn Value"])

In [10]:
X.shape

(7043, 32)

In [11]:
y.shape

(7043,)

In [12]:
X=X.drop(columns=[
    "CustomerID",
    "Count",
    "Churn Label",
    "Churn Score",
    "Churn Reason"
])

In [13]:
X.shape

(7043, 27)

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
X_train, X_test, y_train, y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
X_train.shape

(5634, 27)

In [17]:
X_test.shape

(1409, 27)

In [18]:
X_train.dtypes

Country               object
State                 object
City                  object
Zip Code               int64
Lat Long              object
Latitude             float64
Longitude            float64
Gender                object
Senior Citizen        object
Partner               object
Dependents            object
Tenure Months          int64
Phone Service         object
Multiple Lines        object
Internet Service      object
Online Security       object
Online Backup         object
Device Protection     object
Tech Support          object
Streaming TV          object
Streaming Movies      object
Contract              object
Paperless Billing     object
Payment Method        object
Monthly Charges      float64
Total Charges         object
CLTV                   int64
dtype: object

In [19]:
X_train=X_train.drop(columns=[
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
])

X_test=X_test.drop(columns=[
    "Country",
    "State",
    "City",
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
])

In [20]:
X_train.shape

(5634, 20)

In [21]:
X_train["Total Charges"]=pd.to_numeric(X_train["Total Charges"], errors="coerce")
X_test["Total Charges"]=pd.to_numeric(X_test["Total Charges"], errors="coerce")

In [22]:
X_train["Total Charges"].isnull().sum()

8

In [23]:
train_data=X_train.copy()
train_data["target"]=y_train

train_data=train_data.dropna()

X_train=train_data.drop(columns=["target"])
y_train=train_data["target"]

test_data=X_test.copy()
test_data["target"]=y_test

test_data=test_data.dropna()

X_test=test_data.drop(columns=["target"])
y_test=test_data["target"]

In [24]:
X_train.shape

(5626, 20)

In [25]:
X_train=pd.get_dummies(X_train, drop_first=True)
X_test=pd.get_dummies(X_test, drop_first=True)

In [26]:
X_train, X_test=X_train.align(X_test, join="left", axis=1, fill_value=0)

In [27]:
X_train.shape

(5626, 31)

In [28]:
X_train.select_dtypes(include=["int64", "float64"]).columns

Index(['Tenure Months', 'Monthly Charges', 'Total Charges', 'CLTV'], dtype='object')

In [29]:
X_train=X_train.drop(columns=["CLTV"])
X_test=X_test.drop(columns=["CLTV"])

**Model Comparison**

Several models were evaluated to compare performance:

- Logistic Regression
- Decision Tree (depth=5)
- Random Forest (tuned)
- Gradient Boosting

The following table summarizes their performance on the test set.


In [30]:
comparison=pd.DataFrame({
    "Model": [
        "Logistic (0.4)",
        "Decision Tree (depth=5)",
        "Random Forest (tuned)",
        "Gradient Boosting"
    ],
    "Accuracy": [
        0.773,   
        0.787,
        0.810,
        0.803
    ],
    "Precision": [
        0.556,
        0.594,
        0.676,
        0.659
    ],
    "Recall": [
        0.604,
        0.631,
        0.548,
        0.537
    ],
    "F1": [
        0.579,
        0.612,
        0.606,
        0.592
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1
0,Logistic (0.4),0.773,0.556,0.604,0.579
1,Decision Tree (depth=5),0.787,0.594,0.631,0.612
2,Random Forest (tuned),0.810,0.676,0.548,0.606
3,Gradient Boosting,0.803,0.659,0.537,0.592


Ensemble models achieved higher accuracy. However, Decision Trees and Logistic Regression demonstrated stronger recall- which is critical for churn identification.

**Final Model Selection**

Logistic Regression with a 0.4 classification threshold was selected based on business priorities. 
Since false negatives (missed churners) are more costly than false positives, recall was prioritized.
Although ensemble models achieved competitive accuracy, Logistic Regression achieved competitive recall after threshold adjustment while offering greater interpretability and stability compared to tree-based models.

**Decision Tree vs Logistic Regression**

Decision Trees achieved higher recall when depth was constrained, but exhibited higher variance when fully grown. Logistic Regression demonstrated more stable generalization and stronger interpretability.

**Random Forest: Tuned vs Untuned**

Hyperparameter tuning improved Random Forest recall from ~0.52 to ~0.55, but it remained below Logistic Regression at a business-optimized threshold.

In [31]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

num_cols=X_train.select_dtypes(include=['int64','float64']).columns.tolist()

preprocessor=ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols)
    ]
)

In [32]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe=Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

In [33]:
pipe.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_pipe=pipe.predict(X_test)

accuracy_pipe=accuracy_score(y_test, y_pred_pipe)
precision_pipe=precision_score(y_test, y_pred_pipe)
recall_pipe=recall_score(y_test, y_pred_pipe)
f1_pipe=f1_score(y_test, y_pred_pipe)

accuracy_pipe, precision_pipe, recall_pipe, f1_pipe

(0.7731152204836416,
 0.6053639846743295,
 0.42245989304812837,
 0.49763779527559054)

In [35]:
print(X_train.shape)

(5626, 30)


In [36]:
from sklearn.metrics import roc_auc_score

y_prob_pipe=pipe.predict_proba(X_test)[:, 1]
roc_auc_score(y_test, y_prob_pipe)

0.8114351448824773

In [37]:
y_pred_pipe_40=(y_prob_pipe >= 0.4).astype(int)

precision_40_pipe=precision_score(y_test, y_pred_pipe_40)
recall_40_pipe=recall_score(y_test, y_pred_pipe_40)
f1_40_pipe=f1_score(y_test, y_pred_pipe_40)

precision_40_pipe, recall_40_pipe, f1_40_pipe

(0.5566502463054187, 0.6042780748663101, 0.5794871794871795)

In [38]:
from sklearn.model_selection import cross_val_score

cv_scores_pipe=cross_val_score(pipe, X_train, y_train, cv=5, scoring='recall')

cv_scores_pipe.mean(), cv_scores_pipe.std()

(0.4494983277591973, 0.020896989099415807)

**Cross-Validation**

5-fold cross-validation was performed on the training set using the pipeline to prevent data leakage. The recall scores across folds were stable, indicating consistent generalization performance.

**Business Cost of False Positives vs False Negatives**

In churn prediction, false negatives are more costly than false positives. A false negative means we fail to identify a customer who is about to churn, resulting in lost revenue and missed retention opportunities.
False positives mean we may offer retention incentives to customers who would have stayed anyway. While this has a cost, it is generally lower than the cost of losing a customer entirely.
Given this, we prioritized improving recall to reduce false negatives. Since the model’s (Logistic Regression) AUC was high, the issue lied more with threshold selection than model capability. Therefore, we had adjusted the classification threshold to better align with business goals.

In [39]:
import joblib

In [40]:
joblib.dump(pipe, "../models/churn_model.pkl")

['../models/churn_model.pkl']

In [41]:
loaded_model=joblib.load("../models/churn_model.pkl")

In [42]:
test_pred=loaded_model.predict(X_test)
test_pred[:5]

array([0, 1, 0, 0, 0], dtype=int64)